# 🎬 Movie Matrix AI — Agentic Sistem
**LangGraph + TMDB 5000 Dataset + ML Models**

> Bu notebook, Movie Matrix AI'ın agentic mimarisini Colab'da çalıştırır.
> Sistem her başlangıçta `system_prompt.md`'yi okuyarak görevini öğrenir.

```
Kullanıcı Sorusu
      ↓
  Orchestrator (LangGraph)
      ↓
 ┌────┴────────────────┐
 ↓                    ↓         ↓
search_agent   analysis_agent  recommendation_agent
 ↓
prediction_agent
 ↓
Yanıt
```

## 📦 1. Kurulum

In [ ]:
# Gerekli kütüphaneleri kur
!pip install langgraph langchain-core scikit-learn pandas numpy requests plotly -q
print('✅ Kurulum tamamlandı!')

## 📁 2. Dosyaları Colab'a Yükle

> **Seçenek A:** Google Drive'dan bağla (önerilen)  
> **Seçenek B:** Manuel dosya yükle

In [ ]:
import os

# ── Seçenek A: Google Drive ────────────────────────────────────────────────
USE_DRIVE = True  # Drive kullanmak istemiyorsan False yap

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Drive'daki klasör yolunu buraya gir
    PROJECT_PATH = '/content/drive/MyDrive/movie_matrix_ai'  # ← Değiştir
    os.chdir(PROJECT_PATH)
    print(f'📂 Çalışma dizini: {os.getcwd()}')
else:
    # ── Seçenek B: Manuel Yükleme ──────────────────────────────────────────
    from google.colab import files
    print('Aşağıdaki dosyaları yükle:')
    print('  • tmdb_5000_movies.csv')
    print('  • tmdb_5000_credits.csv')
    print('  • system_prompt.md')
    print('  • agents/ klasörü (zip olarak sıkıştırıp yükleyebilirsin)')

In [ ]:
# Dosya kontrolü
required = [
    'tmdb_5000_movies.csv',
    'tmdb_5000_credits.csv',
    'system_prompt.md',
    'agents/orchestrator.py',
    'agents/search_agent.py',
    'agents/prediction_agent.py',
    'agents/analysis_agent.py',
    'agents/recommendation_agent.py',
]

all_ok = True
for f in required:
    exists = os.path.exists(f)
    status = '✅' if exists else '❌'
    print(f'{status} {f}')
    if not exists:
        all_ok = False

if all_ok:
    print('\n🎉 Tüm dosyalar mevcut!')
else:
    print('\n⚠ Eksik dosyaları yükleyin.')

## 🧠 3. Sistem Promptunu Oku

In [ ]:
# Her başlangıçta sistem görevini öğren
with open('system_prompt.md', 'r', encoding='utf-8') as f:
    SYSTEM_PROMPT = f.read()

print('📋 Sistem Promptu Yüklendi:')
print('=' * 50)
# İlk 500 karakter göster
print(SYSTEM_PROMPT[:500] + '...')
print('=' * 50)
print(f'\n✅ Toplam: {len(SYSTEM_PROMPT)} karakter, {len(SYSTEM_PROMPT.splitlines())} satır')

## 📊 4. Veriyi Yükle & ML Modellerini Eğit

In [ ]:
import ast
import warnings
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier

warnings.filterwarnings('ignore')

print('📂 Veri yükleniyor...')

def load_data():
    m = pd.read_csv('tmdb_5000_movies.csv')
    c = pd.read_csv('tmdb_5000_credits.csv')
    df = m.merge(c, on='title')
    df = df[(df['budget'] > 0) & (df['revenue'] > 0)].reset_index(drop=True)

    def parse_json_col(obj):
        try: return [i['name'] for i in ast.literal_eval(obj)]
        except: return []

    def parse_crew_role(obj, job_filter):
        try: return [i['name'] for i in ast.literal_eval(obj) if i.get('job') == job_filter]
        except: return []

    def parse_cast(obj, limit=5):
        try: return [i['name'] for i in ast.literal_eval(obj)][:limit]
        except: return []

    df['genres_list']    = df['genres'].apply(parse_json_col)
    df['keywords_list']  = df['keywords'].apply(parse_json_col)
    df['companies_list'] = df['production_companies'].apply(parse_json_col)
    df['cast_list']      = df['cast'].apply(lambda x: parse_cast(x, 5))
    df['director_list']  = df['crew'].apply(lambda x: parse_crew_role(x, 'Director'))
    df['director']       = df['director_list'].apply(lambda x: x[0] if x else 'Unknown')

    df['roi']           = df['revenue'] / df['budget']
    df['profit']        = df['revenue'] - df['budget']
    df['success_class'] = df['roi'].apply(lambda x: 2 if x > 2 else (1 if x > 1 else 0))
    df['decade']        = (
        pd.to_datetime(df['release_date'], errors='coerce').dt.year // 10 * 10
    ).astype('Int64').astype(str) + 's'

    # Feature engineering
    df['genre_count'] = df['genres_list'].apply(len)
    dir_roi = df[df['roi'].between(0, 1000)].groupby('director')['roi'].mean()
    df['director_avg_roi'] = df['director'].map(dir_roi).fillna(dir_roi.median())
    cast_rev = df.explode('cast_list').groupby('cast_list')['revenue'].mean()
    df['cast_power'] = df['cast_list'].apply(
        lambda cl: np.mean([cast_rev.get(c, 0) for c in cl]) if cl else 0
    )
    return df

df = load_data()
print(f'✅ {len(df):,} film yüklendi')
print(f'   Sütunlar: {list(df.columns[:8])}')

In [ ]:
# ML Modellerini eğit
print('🤖 ML modelleri eğitiliyor...')

FEATURE_COLS = [
    'budget', 'runtime', 'popularity', 'vote_count',
    'vote_average', 'genre_count', 'director_avg_roi', 'cast_power'
]

X = df[FEATURE_COLS].fillna(0)

reg_model = GradientBoostingRegressor(
    n_estimators=250, learning_rate=0.08, max_depth=5,
    subsample=0.8, random_state=42
).fit(X, df['revenue'])

clf_model = RandomForestClassifier(
    n_estimators=250, max_depth=8, random_state=42
).fit(X, df['success_class'])

print('✅ GradientBoostingRegressor (gişe tahmini) hazır')
print('✅ RandomForestClassifier (başarı sınıfı) hazır')
print('\n🎉 Sistem başlatmaya hazır!')

## 🕸️ 5. LangGraph Agent Grafiğini Oluştur

In [ ]:
import sys
sys.path.insert(0, '.')

from agents.orchestrator import build_graph, run_agent, load_system_prompt

# LangGraph grafini oluştur
graph = build_graph()

if graph:
    print('✅ LangGraph StateGraph oluşturuldu!')
    print('\nAkış:')
    print('  router → [search → predict] | [analyze] | [recommend] → respond → END')
else:
    print('⚠ LangGraph yüklü değil, rule-based fallback aktif')

## 💬 6. Agent ile Konuş

> `query` değişkenini değiştirerek farklı sorgular dene!

In [ ]:
# ── Örnek Sorgular ────────────────────────────────────────────
# Film arama + tahmin:
#   query = "Inception"
#   query = "Interstellar filmini analiz et"
#   query = "Oppenheimer başarılı mı?"

# Dataset analizi:
#   query = "En iyi 10 film hangileri?"
#   query = "Yönetmen Christopher Nolan istatistikleri"
#   query = "Action türü ortalama gişe"
#   query = "Genel dataset özeti"

# Film önerisi:
#   query = "Ne izlesem?"
#   query = "Aksiyon dolu bir şey önerin"
#   query = "Bu akşam korku filmi izlemek istiyorum"
# ─────────────────────────────────────────────────────────────

query = "Inception"

print(f'🔍 Sorgu: {query}')
print('=' * 60)

response = run_agent(
    query=query,
    df=df,
    reg_model=reg_model,
    clf_model=clf_model,
    graph=graph,
)

print(response)

In [ ]:
# Dataset analizi örneği
query = "Genel dataset özeti"

print(f'🔍 Sorgu: {query}')
print('=' * 60)

response = run_agent(
    query=query,
    df=df,
    reg_model=reg_model,
    clf_model=clf_model,
    graph=graph,
)

print(response)

In [ ]:
# Film önerisi örneği
query = "Akşam için aksiyon dolu bir film önerin"

print(f'🔍 Sorgu: {query}')
print('=' * 60)

response = run_agent(
    query=query,
    df=df,
    reg_model=reg_model,
    clf_model=clf_model,
    graph=graph,
)

print(response)

## 🔁 7. İnteraktif Döngü (Sohbet Modu)

> Hücreyi çalıştır ve sorularını yaz. `çıkış` veya `exit` yazarak bitir.

In [ ]:
print('🎬 Movie Matrix AI — Sohbet Modu')
print('Çıkmak için: "çıkış" veya "exit" yaz')
print('=' * 60)

while True:
    try:
        user_input = input('\n💬 Siz: ').strip()
    except (EOFError, KeyboardInterrupt):
        print('\n👋 Görüşmek üzere!')
        break

    if not user_input:
        continue

    if user_input.lower() in ('çıkış', 'exit', 'quit', 'q'):
        print('👋 Görüşmek üzere!')
        break

    print('\n🤖 AI:')
    print('-' * 40)

    response = run_agent(
        query=user_input,
        df=df,
        reg_model=reg_model,
        clf_model=clf_model,
        graph=graph,
    )

    print(response)
    print('-' * 40)

## 🏗️ 8. Agent Mimarisi Görselleştirme

In [ ]:
# Sistemin nasıl çalıştığını görselleştir
print("""
╔══════════════════════════════════════════════════╗
║          MOVIE MATRIX AI — AGENT MİMARİSİ        ║
╚══════════════════════════════════════════════════╝

  📋 system_prompt.md  ←  Her başlangıçta okunur
          ↓
  👤 Kullanıcı Sorusu
          ↓
  🕸️  Orchestrator (LangGraph StateGraph)
          ↓
     [Intent Router]
     ┌────┼─────────────────┐
     ↓    ↓                 ↓
  🔍 Search  📊 Analysis  🍿 Recommendation
  (TMDB API) (Dataset)    (Filtre + Ruh Hali)
     ↓
  🤖 Prediction
  (GBR + RF Modeli)
     ↓
  ✅ Final Response

Dataset: TMDB 5000 ({:,} film)
Modeller: GradientBoostingRegressor + RandomForestClassifier
Features: budget, runtime, popularity, vote_count,
          vote_average, genre_count, director_avg_roi, cast_power
""".format(len(df)))

---
## 📌 Not: Docker Hakkında

Docker bu notebook'ta kullanılmıyor çünkü **Colab'da Docker daemon çalışmaz**.

Ancak local geliştirmede Docker ile çalıştırmak için proje kökünde `Dockerfile` ve `docker-compose.yml` dosyaları mevcuttur:

```bash
# Local'de Docker ile çalıştırma (Colab'da değil, bilgisayarında)
docker-compose up

# Bu komut:
# • Streamlit uygulamasını (app.py) 8501 portunda başlatır
# • FastAPI'yi (api.py) 8000 portunda başlatır
```

**Colab için:** Bu notebook yeterli. Tüm agentic mantık burada çalışır.